# Recommended Environment Google Colab GPU
This is codebase for DTWMatching with prototypical learning, used to train a model capable of continous and zero-shot sign language interpretation from skeletal data.

Input for the model is T x 42 x 3 where T is the temporal length.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Declare root directories
root_dir = '/content/drive/MyDrive/SignData/test'
zero_shot_root_dir = '/content/drive/MyDrive/SignData/zeroshotTest'

In [ ]:
!pip install textdistance

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.transform import Slerp, Rotation
from scipy.interpolate import interp1d
import torch.nn.functional as F

class HandGestureDataset(Dataset):
    def __init__(self, root_dir, num_classes_per_batch, num_distractions_per_batch, max_sequence_length, batch_length=100, mirror = True, speed_variation=0.5, random_slice=True, sample_all=False, train=True):
        self.root_dir = root_dir
        self.num_classes_per_batch = num_classes_per_batch
        self.num_distractions_per_batch = num_distractions_per_batch
        self.max_sequence_length = max_sequence_length
        self.speed_variation = speed_variation
        self.random_slice = random_slice
        self.batch_length = batch_length
        self.sample_all = sample_all
        self.train = train
        self.mirror = mirror

        self.class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {}
        self.data_files = {}
        for i, class_name in enumerate(self.class_names):
            class_dir = os.path.join(root_dir, class_name)
            self.data_files[class_name] = sorted(os.listdir(class_dir))
            self.class_to_idx[class_name] = i

        self.train_files = {}
        self.val_files = {}
        for class_name, files in self.data_files.items():
            split_idx = int(0.8 * len(files))
            self.train_files[class_name] = files[:split_idx]
            self.val_files[class_name] = files[split_idx:]

    def __len__(self):
        return self.batch_length

    def vary_speed(self, gesture_data):
        speed_factor = np.random.uniform(1 - self.speed_variation, 1 + self.speed_variation)

        original_frames = gesture_data.shape[0]
        new_frames = int(original_frames / speed_factor)

        old_times = np.arange(original_frames)
        new_times = np.linspace(0, original_frames - 1, new_frames)

        interpolator = interp1d(old_times, gesture_data, axis=0, kind='linear')
        new_gesture_data = interpolator(new_times)

        return new_gesture_data

    def vary_position(self, gesture_data):
      return gesture_data + np.random.uniform(-0.05, 0.05, (3,))

    def concatenate_hand_gestures(self, gesture_data_list, class_indices):
        concatenated_data = []
        concatenated_labels = []
        transition_frames_length = np.random.randint(5, 30, size=len(gesture_data_list)-1)

        for i, (current_data, class_idx) in enumerate(zip(gesture_data_list, class_indices)):
            concatenated_data.append(current_data)
            concatenated_labels.extend([class_idx] * len(current_data))

            if i < len(gesture_data_list) - 1:
                next_data = gesture_data_list[i + 1]

                end_frame = current_data[-1]
                start_frame = next_data[0]

                transition_data = []

                for j in range(42):
                    rotations = Rotation.from_rotvec(np.vstack([end_frame[j], start_frame[j]]))

                    slerp = Slerp(times=[0, 1], rotations=rotations)

                    transition_times = np.linspace(0, 1, transition_frames_length[i])
                    landmark_transition = slerp(transition_times).as_rotvec()

                    transition_data.append(landmark_transition)

                transition_data = np.transpose(np.array(transition_data), (1, 0, 2))

                concatenated_data.append(transition_data)
                concatenated_labels.extend([-1] * transition_frames_length[i])

        concatenated_data = np.vstack(concatenated_data)
        concatenated_labels = np.array(concatenated_labels)

        return concatenated_data, concatenated_labels

    def __getitem__(self, index):
        file_set = self.train_files if self.train else self.val_files

        input_classes = random.sample(self.class_names, self.num_classes_per_batch)

        gesture_data_list = []
        class_indices = []
        for class_name in input_classes:
            data_file = random.choice(file_set[class_name])
            data_path = os.path.join(self.root_dir, class_name, data_file)
            gesture_data = np.load(data_path)
            gesture_data = self.vary_speed(self.vary_position(gesture_data))
            if self.mirror:
              gesture_data[:,:,0] *= -1
              gesture_data[:,:,0] += 1
            gesture_data_list.append(gesture_data)
            class_indices.append(self.class_to_idx[class_name])

        input_sequence, input_label = self.concatenate_hand_gestures(gesture_data_list, class_indices)

        if self.random_slice:
            start_idx = random.randint(0, min(15, len(input_sequence)//2))
            end_idx = len(input_sequence)-random.randint(0, min(15, len(input_sequence)//2))
            input_sequence = input_sequence[start_idx:end_idx]
            curr_idx = start_idx
            prev=input_label[start_idx]
            while input_label[curr_idx] == prev:
              input_label[curr_idx] = -2 #-2 represents sliced inputs. The model will ignore these
              curr_idx += 1

            curr_idx = end_idx
            prev=input_label[end_idx - 1]
            while input_label[curr_idx - 1] == prev:
              input_label[curr_idx - 1] = -2
              curr_idx -= 1

            input_label = input_label[start_idx:end_idx]


        input_sequence = torch.from_numpy(input_sequence).float()
        input_label = torch.from_numpy(input_label).long()

        additional_classes = random.choices(self.class_names, k=self.num_distractions_per_batch)
        sample_classes = list(set(input_classes + additional_classes))
        if self.sample_all:
          sample_classes = self.class_names

        sample_sequences = []
        sample_labels = []
        for class_name in sample_classes:
            data_file = random.choice(file_set[class_name])
            data_path = os.path.join(self.root_dir, class_name, data_file)
            gesture_data = np.load(data_path)
            gesture_data = self.vary_speed(self.vary_position(gesture_data))
            if self.mirror:
              gesture_data[:,:,0] *= -1
              gesture_data[:,:,0] += 1
            sample_sequences.append(torch.from_numpy(gesture_data).float())
            sample_labels.append(self.class_to_idx[class_name])

        max_length = max(seq.shape[0] for seq in sample_sequences)

        padded_sequences = []
        sample_masks = []
        for seq in sample_sequences:
            padding_length = max_length - seq.shape[0]
            padded_seq = F.pad(seq.permute(1, 2, 0), (0, padding_length), mode='constant', value=0)
            padded_sequences.append(padded_seq.permute(2, 0, 1))

            mask = torch.ones(max_length, dtype=torch.bool)
            mask[seq.shape[0]:] = 0
            sample_masks.append(mask)

        sample_sequences = torch.stack(padded_sequences)
        sample_labels = torch.tensor(sample_labels)
        sample_masks = torch.stack(sample_masks)

        return input_sequence, input_label, sample_sequences, sample_labels, sample_masks

random.seed(42)
np.random.seed(42)
torch.random.manual_seed(42)




train_dataset = HandGestureDataset(root_dir, 10, 10, 150, train=True)
val_dataset = HandGestureDataset(root_dir, 10, 10, 150, batch_length=10, train=False)


train_loader = DataLoader(train_dataset, shuffle=True, num_workers=2, batch_size=1)
val_loader = DataLoader(val_dataset, shuffle=False, num_workers=2, batch_size=1)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

class spatialExtractor(nn.Module):
    def __init__(self, input_channels, hidden_channels, output_channels):
        super(spatialExtractor, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(hidden_channels)
        self.conv2 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(hidden_channels)
        self.conv3 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(hidden_channels)
        self.conv4 = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(hidden_channels)
        self.conv5 = nn.Conv2d(hidden_channels, output_channels, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(output_channels)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        return x

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(n_outputs)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(n_outputs)
        self.relu2 = nn.ReLU()

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class Model(nn.Module):
    def __init__(self, embedding_dim=256, num_landmarks=42, input_channels=3, input_frames=30):
        super(Model, self).__init__()

        self.spatial_extractor = spatialExtractor(input_channels, 16, 32)

        self.dim_reduction = nn.Conv1d(32 * num_landmarks, embedding_dim, kernel_size=1)

        num_channels = [embedding_dim, embedding_dim, embedding_dim]
        self.tcn = nn.Sequential(
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[0], num_channels[1], kernel_size=3, stride=1, dilation=1, padding=1),
            TemporalBlock(num_channels[1], num_channels[2], kernel_size=3, stride=1, dilation=2, padding=2),
        )

        self.partition_threshold = nn.Parameter(torch.tensor(5.0))
        self.match_threshold = nn.Parameter(torch.tensor(0.5))


    def forward(self, x):
        batch_size, frames, landmarks, channels = x.size()

        x = x.permute(0, 1, 3, 2).contiguous().view(-1, channels, landmarks, 1)

        x = self.spatial_extractor(x)

        _, features, _, _ = x.size()
        x = x.view(batch_size, frames, features, landmarks).permute(0, 2, 3, 1)

        x = x.contiguous().view(batch_size, -1, frames)

        x = self.dim_reduction(x)

        x = self.tcn(x)


        x = x.permute(0, 2, 1).contiguous()

        return x


In [ ]:
from typing import List, Tuple
from numba import njit
import textdistance

@njit
def dtw_path(x: np.ndarray, y: np.ndarray) -> List[Tuple[int, int]]:
    N, E = x.shape
    M, _ = y.shape

    dtw = np.full((N + 1, M + 1), np.inf)
    dtw[0, 0] = 0.0

    for i in range(1, N + 1):
        for j in range(1, M + 1):
            cost = np.linalg.norm(x[i-1] - y[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j],
                                   dtw[i, j-1],
                                   dtw[i-1, j-1])

    path = []
    i, j = N, M
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            min_cost_index = np.argmin(np.array([dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1]]))
            if min_cost_index == 0:
                i -= 1
            elif min_cost_index == 1:
                j -= 1
            else:
                i -= 1
                j -= 1

    path.reverse()

    return path


def partition_sequence(sequence, labels):
  partitioned = []
  current_label = labels[0]
  current_partition = []

  for i, label in enumerate(labels):
      if label == current_label:
          current_partition.append(sequence[i])
      else:
          partitioned.append((torch.stack(current_partition), current_label))
          current_label = label
          current_partition = [sequence[i]]
  partitioned.append((torch.stack(current_partition), current_label))

  return partitioned

def dtw_partition_loss(input_embedding, target_embeddings, labels, threshold, alpha):
  episodes = partition_sequence(input_embedding, labels)
  total_predictions = 0
  correct_predictions = 0
  loss = 0
  for episode, label in episodes:
      if label == -2:
        continue
      distance = []
      correct_class = None
      for target_label, target_embedding in enumerate(target_embeddings):
        path = dtw_path(episode.detach().cpu().numpy(), target_embedding.detach().cpu().numpy())
        dtw_dist = torch.tensor(0.0, device=input_embedding.device)
        for (i, j) in path:
            dtw_dist += torch.norm(episode[i] - target_embedding[j])
        distance.append(dtw_dist)
        if target_label == label.item():
          correct_class = dtw_dist
      distance.append(threshold)
      output = -torch.stack(distance)
      loss += F.cross_entropy(output, label)
      if correct_class:
        loss += alpha*correct_class


      predicted = torch.argmax(output)
      if predicted == label:
        correct_predictions += 1

      total_predictions += 1



  return loss, correct_predictions, total_predictions





@njit
def subsequence_DTW_helper(x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    N, E = x.shape
    M, _ = y.shape

    dtw = np.empty((N, M))
    origins = np.empty((N, M), dtype=np.int64)

    for i in range(N):
        dist = np.linalg.norm(x[i] - y[0])
        dtw[i, 0] = dist
        origins[i, 0] = i
    for j in range(1, M):
        dist = np.linalg.norm(x[0] - y[j])
        dtw[0, j] = dist + dtw[0, j-1]
        origins[0, j] = 0

    for i in range(1, N):
        for j in range(1, M):
            dist = np.linalg.norm(x[i] - y[j])

            cost_diag = dtw[i-1, j-1]
            cost_left = dtw[i, j-1]
            cost_up = dtw[i-1, j]

            if cost_diag <= cost_left and cost_diag <= cost_up:
                dtw[i, j] = dist + cost_diag
                origins[i, j] = origins[i-1, j-1]
            elif cost_left < cost_up:
                dtw[i, j] = dist + cost_left
                origins[i, j] = origins[i, j-1]
            else:
                dtw[i, j] = dist + cost_up
                origins[i, j] = origins[i-1, j]

    return dtw, origins

@njit
def calculate_path(dtw: np.ndarray, start: int, end: int) -> List[Tuple[int, int]]:
  path = []
  N, M = dtw.shape
  i, j = end, M-1
  while i > start or j > 0:
      path.append((i, j))
      if i == start:
          j -= 1
      elif j == 0:
          i -= 1
      else:
          min_cost_index = np.argmin(np.array([dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1]]))
          if min_cost_index == 0:
              i -= 1
          elif min_cost_index == 1:
              j -= 1
          else:
              i -= 1
              j -= 1
  path.append((start, 0))
  return path


def subsequence_DTW_torch(x, y):
    dtw, origins = subsequence_DTW_helper(x.detach().cpu().numpy(), y.detach().cpu().numpy())

    current_min = torch.tensor(float('inf'))
    min_costs = torch.empty((len(x),), device=x.device)
    curr_origin = origins[-1, -1]

    for i in range(len(x)-1, -1, -1):
        if current_min.item() > dtw[i, -1]:
            path = calculate_path(dtw, origins[i, -1], i)
            dtw_dist = 0
            for (I, J) in path:
                dtw_dist += torch.norm(x[I] - y[J])
            current_min = dtw_dist
            curr_origin = origins[i, -1]
        elif i < curr_origin:
            current_min += torch.norm(x[i] - y[0])
        min_costs[i] = current_min
    return min_costs



def dtw_match_loss(input_embedding, target_embeddings, labels, threshold):
  costs = []

  for target_embedding in target_embeddings:
    DTW_costs = subsequence_DTW_torch(input_embedding, target_embedding)/len(target_embedding)
    costs.append(DTW_costs)
  costs.append(torch.full((len(input_embedding),), threshold.item(), requires_grad=True, device=input_embedding.device))
  _, predictions = torch.max(-torch.stack(costs), dim=0)
  costs = torch.stack(costs).permute((1, 0)).contiguous()
  loss = F.cross_entropy(-costs, labels, ignore_index=-2)

  prevPredict = None
  prevTarget = None
  sentence=[]
  target_sentence=[]
  for i, (label, result) in enumerate(zip(labels.detach().cpu().numpy(), predictions.detach().cpu().numpy())):
    if labels[i] == -2:
      continue

    if prevPredict!=result:
      prevPredict = result
      if result!=len(target_embeddings):
        sentence.append(result)

    if prevTarget!=label:
      prevTarget = label
      if label!=len(target_embeddings):
        target_sentence.append(label)

  accuracy = 1-(textdistance.levenshtein.distance(sentence, target_sentence)/max(len(target_sentence), len(sentence)))

  return loss, accuracy



In [ ]:

def train_protonet(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    alpha = 0.05
    beta = 0.5
    total_accuracy = 0
    total = 0
    for i, batch in enumerate(train_loader):
        t1=time.perf_counter()
        input_sequence, input_label, target_sequences, target_labels, target_masks = [b.to(device)[0] for b in batch]
        label_to_index = {label.item(): idx for idx, label in enumerate(target_labels)}
        label_to_index[-1] = len(target_labels)
        label_to_index[-2] = -2
        labels = torch.empty_like(input_label, device=device)
        for i, value in enumerate(input_label):
            labels[i] = label_to_index.get(value.item(), len(target_labels))

        optimizer.zero_grad()
        input_embedding = model(input_sequence.unsqueeze(0)).squeeze(0)
        target_embeddings = [model(target_sequence[target_mask].unsqueeze(0)).squeeze(0) for target_sequence, target_mask in zip(target_sequences, target_masks)]

        partition_loss, _, _ = dtw_partition_loss(input_embedding, target_embeddings, labels, model.partition_threshold, alpha=alpha)
        match_loss, accuracy = dtw_match_loss(input_embedding, target_embeddings, labels, model.match_threshold)

        loss = partition_loss + match_loss*beta

        total_accuracy += accuracy
        total += 1
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / total

    avg_accuracy = total_accuracy/total

    return avg_loss, avg_accuracy

def evaluate_protonet(model, val_loader, device):
    model.eval()
    total_loss = 0
    alpha = 0.05
    beta = 0.5
    total_accuracy = 0
    total = 0
    with torch.no_grad():
      for i, batch in enumerate(val_loader):
          t1=time.perf_counter()
          input_sequence, input_label, target_sequences, target_labels, target_masks = [b.to(device)[0] for b in batch]
          label_to_index = {label.item(): idx for idx, label in enumerate(target_labels)}
          label_to_index[-1] = len(target_labels)
          label_to_index[-2] = -2
          labels = torch.empty_like(input_label, device=device)
          for i, value in enumerate(input_label):
              labels[i] = label_to_index.get(value.item(), len(target_labels))

          input_embedding = model(input_sequence.unsqueeze(0)).squeeze(0)
          target_embeddings = [model(target_sequence[target_mask].unsqueeze(0)).squeeze(0) for target_sequence, target_mask in zip(target_sequences, target_masks)]

          partition_loss, _, _ = dtw_partition_loss(input_embedding, target_embeddings, labels, model.partition_threshold, alpha=alpha)
          match_loss, accuracy = dtw_match_loss(input_embedding, target_embeddings, labels, model.match_threshold)

          loss = partition_loss + match_loss*beta

          total_accuracy += accuracy
          total += 1
          total_loss += loss.item()

    avg_loss = total_loss / total

    avg_accuracy = total_accuracy/total

    return avg_loss, avg_accuracy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
state_dict = torch.load('/content/drive/MyDrive/SignData/protomodel/model-10.h5', map_location=device)
if 'partition_threshold' not in state_dict.keys():
  state_dict['partition_threshold'] = torch.tensor(4.0)
if 'match_threshold' not in state_dict.keys():
  state_dict['match_threshold'] = torch.tensor(0.3)
state_dict.pop('threshold')
model.load_state_dict(state_dict)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 50
for epoch in range(num_epochs):
    train_loss, train_acc =  train_protonet(model, train_loader, optimizer, device)
    eval_loss, eval_acc = evaluate_protonet(model, val_loader, device)
    torch.save(model.state_dict(), f'/content/drive/MyDrive/SignData/protomodel/model-{epoch+1}.h5')

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Eval Loss: {eval_loss:.4f}, Eval Acc: {eval_acc:.4f}")
    print("-----------------------------")




In [ ]:
torch.save(embedder.state_dict(), f'/content/drive/MyDrive/SignData/protomodel/protoModelFinal.h5')